In [ ]:
import os, io, random, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import models, transforms
from datasets import load_from_disk
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Prediction device: {device}")

IMG_COL = "Image"

def find_dirs(top, names, max_depth=12):
    found, base = [], top.rstrip(os.sep).count(os.sep)
    for root, dirs, _ in os.walk(top):
        if root.count(os.sep) - base >= max_depth:
            dirs[:] = []
            continue
        for d in list(dirs):
            if d.lower() in names:
                found.append(os.path.join(root, d))
            if d.lower() in {"ai", "nature"}:
                dirs.remove(d)
    return found

# 1. Dataset Resolution
saved = find_dirs("/kaggle/input", {"defactify_subset"}, max_depth=12)
if not saved:
    saved = find_dirs("/kaggle/working", {"defactify_subset"}, max_depth=12)
assert saved, "Could not find defactify_subset. Attach the dataset to this notebook."

SAVED_ROOT = os.path.dirname(saved[0])
SPLITS_DIR = os.path.join(SAVED_ROOT, "splits")
defactify = load_from_disk(os.path.join(SAVED_ROOT, "defactify_subset"))
print("Dataset loaded from:", SAVED_ROOT)

# Locate BigGAN images
biggan_sub = os.path.join(SAVED_ROOT, "biggan_subset")
if os.path.exists(biggan_sub):
    BIGGAN_DIR = biggan_sub
else:
    found = find_dirs("/kaggle/input", {"ai", "nature"}, max_depth=12)
    BIGGAN_DIR = os.path.commonpath(found) if found else SAVED_ROOT
    while os.path.basename(BIGGAN_DIR).lower() in {"ai", "nature", "train", "val"}:
        BIGGAN_DIR = os.path.dirname(BIGGAN_DIR)

# 2. Dataset Class
class PredictionImageDataset(Dataset):
    def __init__(self, records, hf_data, local_root, transform=None):
        self.rows = records
        self.hf = hf_data
        self.root = local_root
        self.tf = transform
        
    def __len__(self):
        return len(self.rows)
        
    def __getitem__(self, i):
        r = self.rows[i]
        if r["storage"] == "hf":
            img = self.hf[r["hf_split"]][int(r["hf_index"])][IMG_COL]
        else:
            img = Image.open(os.path.join(self.root, r["image_path"]))
        img = img.convert("RGB")
        raw_img = img.copy()
        tensor = self.tf(img) if self.tf else img
        return tensor, raw_img, int(r["binary_label"]), str(r.get("generator", "Unknown"))

def build_resnet50(num_classes=2):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# 3. Flexible Checkpoint Finding (finds checkpoint anywhere in input or working)
ckpt_candidates = (
    glob.glob("/kaggle/input/**/seen_all_finetune_A_best.pt", recursive=True) +
    glob.glob("/kaggle/input/**/seen_all_frozen_A_best.pt", recursive=True) +
    ["/kaggle/working/AI_Image_Generalization/checkpoints/seen_all_finetune_A_best.pt",
     "/kaggle/working/AI_Image_Generalization/checkpoints/seen_all_frozen_A_best.pt"]
)
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)
assert ckpt_path is not None, "No trained checkpoint found. Ensure the previous notebook output is attached."

model = build_resnet50(2).to(device)
state = torch.load(ckpt_path, map_location=device, weights_only=False)
if isinstance(state, dict) and "model" in state:
    state = state["model"]
model.load_state_dict(state)
model.eval()
print("Loaded trained checkpoint:", ckpt_path)

# 4. Evaluation Transform
class JPEGRecompress:
    def __init__(self, quality=90):
        self.quality = quality
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=self.quality)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
eval_tf = transforms.Compose([
    transforms.CenterCrop(128),
    JPEGRecompress(90),
    transforms.ToTensor(),
    norm
])

test_csv = os.path.join(SPLITS_DIR, "seen_all", "test.csv")
test_df = pd.read_csv(test_csv)

# 5. Plotting Function
def show_generator_predictions(generator_name, n_each=3, seed=42):
    fake_df = test_df[
        (test_df["binary_label"] == 1) &
        (test_df["generator"].astype(str) == generator_name)
    ]
    if fake_df.empty:
        print(f"No test images found for generator: {generator_name}")
        return

    real_source = "ImageNet_real" if generator_name == "BigGAN" else "Defactify_real"
    real_df = test_df[
        (test_df["binary_label"] == 0) &
        (test_df["source_dataset"].astype(str) == real_source)
    ]
    if real_df.empty:
        real_df = test_df[test_df["binary_label"] == 0]

    n_real = min(n_each, len(real_df))
    n_fake = min(n_each, len(fake_df))
    real_sample = real_df.sample(n=n_real, random_state=seed)
    fake_sample = fake_df.sample(n=n_fake, random_state=seed)
    records = real_sample.to_dict("records") + fake_sample.to_dict("records")
    ds = PredictionImageDataset(records, defactify, BIGGAN_DIR, transform=eval_tf)

    n = len(ds)
    fig, axes = plt.subplots(n, 1, figsize=(10, max(4, n * 4.2)))
    if n == 1:
        axes = [axes]
    labels_map = {0: "REAL", 1: "FAKE"}

    for i, ax in enumerate(axes):
        img_tensor, raw_img, true_label, source_generator = ds[i]
        with torch.inference_mode():
            logits = model(img_tensor.unsqueeze(0).to(device))
            prob_fake = torch.softmax(logits.float(), dim=1)[0, 1].item()

        pred_label = int(prob_fake >= 0.5)
        confidence = prob_fake if pred_label == 1 else (1.0 - prob_fake)
        correct = (pred_label == true_label)
        status = "CORRECT" if correct else "INCORRECT"

        ax.imshow(raw_img)
        ax.axis("off")
        ax.set_title(
            f"Predicted: {labels_map[pred_label]} ({confidence:.1%} confidence)   |   "
            f"Actual: {labels_map[true_label]}   |   {status}\n"
            f"Dataset generator/source: {source_generator}",
            fontsize=12,
            color=("green" if correct else "red"),
            pad=10
        )

    fig.suptitle(f"Real vs {generator_name}: model predictions on test-set images",
                 fontsize=16, fontweight="bold", y=1.0)
    plt.tight_layout()
    plt.show()

In [ ]:
import os, json, glob
import pandas as pd

# 1. Define expected experiments and modes
EXPS = [
    "seen_all", "gan_to_diffusion", "diffusion_to_gan",
    "unseen_BigGAN", "unseen_DALL_E_3", "unseen_Midjourney_v6",
    "unseen_Stable_Diffusion_2_1", "unseen_Stable_Diffusion_3", "unseen_Stable_Diffusion_XL"
]
MODES = ["frozen", "finetune"]
EXPECTED_TAGS = [f"{exp}_{mode}_A" for mode in MODES for exp in EXPS]

# 2. Locate directory containing outputs
base_dir = "/kaggle/input/notebooks/eshanm/02-train/AI_Image_Generalization"
ckpt_dir = f"{base_dir}/checkpoints"
res_dir = f"{base_dir}/results"

# 3. Check checkpoints and JSON results
rows = []
for tag in EXPECTED_TAGS:
    best_pt = os.path.exists(f"{ckpt_dir}/{tag}_best.pt")
    last_pt = os.path.exists(f"{ckpt_dir}/{tag}_last.pt")
    json_path = f"{res_dir}/{tag}.json"
    json_exists = os.path.exists(json_path)
    
    auc = None
    if json_exists:
        try:
            data = json.load(open(json_path))
            auc = data.get("auc", None)
        except Exception:
            pass

    rows.append({
        "experiment_tag": tag,
        "best_checkpoint": "✅" if best_pt else "❌",
        "last_checkpoint": "✅" if last_pt else "❌",
        "result_json": "✅" if json_exists else "❌",
        "test_auc": round(auc, 4) if auc is not None else "N/A"
    })

df_status = pd.DataFrame(rows)
display(df_status)

# 4. Summary Verdict
total_expected = len(EXPECTED_TAGS)
completed = sum(df_status["result_json"] == "✅")
checkpoints_saved = sum(df_status["best_checkpoint"] == "✅")

print(f"\n--- Run Summary ---")
print(f"Total experiments configured: {total_expected}")
print(f"Completed with JSON results: {completed} / {total_expected}")
print(f"Best checkpoints verified:    {checkpoints_saved} / {total_expected}")

if completed == total_expected and checkpoints_saved == total_expected:
    print("\n🎉 All 18 models successfully ran and saved both results and checkpoints!")
else:
    print("\n⚠️ Some models are missing. Check rows marked with ❌ above.")

In [ ]:
show_generator_predictions("BigGAN", n_each=3)

In [ ]:
show_generator_predictions("DALL_E_3", n_each=3)

In [ ]:
show_generator_predictions("Midjourney_v6", n_each=3)

In [ ]:
show_generator_predictions("Stable_Diffusion_2_1", n_each=3)

In [ ]:
show_generator_predictions("Stable_Diffusion_3", n_each=3)

In [ ]:
show_generator_predictions("Stable_Diffusion_XL", n_each=3)